In [18]:
import sys
import os

# This adds the parent directory (..) to the Python path
sys.path.append(os.path.abspath('..'))

from datasets import load_dataset
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from typing import List

from transformers import (
    AutoModel,
    AutoTokenizer,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    AutoConfig,
    set_seed,
)

from models import CloseTrack_Predictor

from utils import ( 
    compute_metrics, 
    load_model_params,
    load_data_paths,
    preprocess_dataset,
    cleanup_trainer_memory
)

### Download Model

First, download the model from my Huggingface hub to the folder "../models/model_name"

In [2]:
import logging
from pathlib import Path
from huggingface_hub import snapshot_download
from utils import is_model_downloaded

MODELS_DIR = Path("../models/")
HF_USER = "HenryLi0925"

def download_models(models_to_run):
    """
    Download models from Hugging Face Hub to local folder.

    Args:
        models_to_run (list[str]): List of model names to download.
    """

    for model_name in models_to_run:
        
        dest_folder = MODELS_DIR / model_name
        dest_folder.mkdir(parents=True, exist_ok=True)

        if is_model_downloaded(dest_folder):
            logging.info(f"{model_name} already downloaded, skipping.")
            continue

        logging.info(f"Downloading {model_name}...")    
        
        # Download all files from the HF repo into the local folder
        snapshot_download(
            repo_id=f"{HF_USER}/{model_name}",
            local_dir=str(dest_folder),
            local_dir_use_symlinks=False
        )
        logging.info(f"Saved {model_name} to {dest_folder}\n")

In [6]:
models_to_download = ["ScalarMix_AddAttn_mlp", "ScalarMix_SelfAttn_mlp"]
download_models(models_to_download)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/739 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

### Load Model

Load the model using the following commands. NOTE: do not change the argument that i specified for each model, otherwise the required components won't be correctly instantiated. 

In [8]:
DATA_DIR = Path("../data/")
MODELS_DIR = Path("../models/")
ScalarMixing_AddAttn_modelpath = MODELS_DIR / "ScalarMix_AddAttn_mlp"
ScalarMixing_SelfAttn_modelpath = MODELS_DIR / "ScalarMix_SelfAttn_mlp"

In [9]:
ScalarMixing_AddAttn_model = CloseTrack_Predictor.from_pretrained(
    ScalarMixing_AddAttn_modelpath,
    layer_wise = 'ScalarMix',
    token_wise = 'AddAttn'
)

In [10]:
ScalarMixing_SelfAttn_model = CloseTrack_Predictor.from_pretrained(
    ScalarMixing_SelfAttn_modelpath,
    layer_wise = 'ScalarMix',
    token_wise = 'SelfAttn'
)

### Load data

You can load some data and inspect the prediction from the trained model.

In [13]:
data_files = load_data_paths(DATA_DIR, 'xx', "finetune")
hf_dataset = load_dataset("csv", data_files=data_files)
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base', use_fast=True)
cols_to_merge = "L1_source_word; L1_context; en_target_clue; en_target_word".split("; ")
sep_token = f" {tokenizer.sep_token} " if tokenizer.sep_token else " "
preprocessed_ds = preprocess_dataset(hf_dataset, cols_to_merge, sep_token)

# Tokenize dataset
tokenized_ds = preprocessed_ds.map(
    lambda x: tokenizer(x["input_text"], truncation=True),
    batched=True,
    desc="Tokenizing input text"
)
# Itemize L1 
l1_to_idx = {l1:i for i, l1 in enumerate(["es", "de", "cn"])}
tokenized_ds = tokenized_ds.map(
    lambda x: {"l1_encode": [l1_to_idx[val] for val in x['L1']]},
    batched=True,
    remove_columns=['L1'],
    desc="Itemizing L1"
)

# Initialize the collator with your tokenizer
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Train dataset and dataloader
train_ds = tokenized_ds['train']
train_dataloader = DataLoader(train_ds.remove_columns(['input_text']), batch_size=64, collate_fn = data_collator)

In [14]:
# fetch a batch of data
data = next(iter(train_dataloader))

### Model Inspection

Inspect the learned parameters in the ScalarMixing module.

##### ScalarMixing module

First, we learn a temperature parameter $\tau$ which is transformed from the original parameter $t$ through the following formula:

$$
\tau = w + c * t
$$

where $w$ and $c$ are constant picked for base temp and factor, respectively. In this case, we choose $w = 10^2$ and $c = 10^5$

Then, the scalar weight $w_l$ for each layer $l$ is multiplied by the temperature $\tau$, before being normalized to $z_l$.

$$
z_l = \text{Softmax}(\tau * w_l)
$$


Additional strategy to ensure numerical stability during optimization is to perform layer normalization on hidden states across layers. Each layer's hidden states are then weighted averaged by the global scalar weights and finally multiplied by the $\gamma$ factor:

$$
S(h_0, \dots, h_{L}) = \gamma \cdot \sum_{l=0}^L z_l \cdot \text{LayerNorm}(h_l)
$$



After evaluation on multiple strategies, the final scalar mixing modulde that i choose have the following setup:
- do_layer_norm: `True`, perform layer normalization to hidden states.
- have_temperature: `True`, regulate the scalar weights with the additional temperature parameter.
- trainable: `True`, fix $\gamma$ to 1 rathen than make it trainable. 

**Problem**: a look at the scalar weights suggests that the learned weights explode for some layers while vainishing for the other. Making the problem even worse, one layer sticks out with a scalar weight that is larger than others by magnitude, resulting in a numerically 1-0 weights after softmaxed, which is unintended because we want to use Scalar Mixing to learn a smooth distribution of weights across layers.

In [15]:
for name, params in ScalarMixing_AddAttn_model.named_parameters():
    if "ScalarMix" in name:
        print(f"{name}: {params}")

ScalarMix.temp: Parameter containing:
tensor([3.6861e-29], requires_grad=True)
ScalarMix.scalar_weights: Parameter containing:
tensor([1.7994e-29, 0.0000e+00, 0.0000e+00, 0.0000e+00, 7.3303e+22, 3.9895e-11,
        7.1855e+22, 7.1955e+28, 7.1481e+22, 2.0997e+26, 3.5243e+33, 4.3534e+24,
        1.8006e-29], requires_grad=True)


In [22]:
ScalarMixing_AddAttn_model.ScalarMix.scalar_weights * tau

tensor([1.7994e-27, 0.0000e+00, 0.0000e+00, 0.0000e+00, 7.3303e+24, 3.9895e-09,
        7.1855e+24, 7.1955e+30, 7.1481e+24, 2.0997e+28, 3.5243e+35, 4.3534e+26,
        1.8006e-27], grad_fn=<MulBackward0>)

In [24]:
base_temp = 1e2
factor = 1e5
tau = base_temp + factor * ScalarMixing_AddAttn_model.ScalarMix.temp
F.softmax(ScalarMixing_AddAttn_model.ScalarMix.scalar_weights * tau, dim=0)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       grad_fn=<SoftmaxBackward0>)

##### Attention module

Because it dynamically calculate the attention score for each sequence, we have to feed in a data batch to inspect its output hidden states and attention weights.

In [16]:
output = ScalarMixing_AddAttn_model(*(data['input_ids'], data['attention_mask'], data['labels']), output_attentions=True)
attention_masks = data['attention_mask']
logits, hiddens, attentions, attn_weights = output['logits'], output['hidden_states'], output['attentions'], output['token_attn_weights']
hiddens = torch.stack(hiddens,0)

Right now i am not sure if my attention implemenation in the model is right by simply looking at these outputs. But the results are not ideal since the evaluation results on these attention-based regressors did not outperform the baseline. You can take a look at the `models.py`.

In [27]:
# the input sequence of the first sample in our batch
tokenizer.decode(data['input_ids'][0])

'<s> lapso</s> El eclipse solar fue visible durante un breve lapso de tiempo.</s> s___</s> span</s><pad><pad><pad><pad><pad><pad><pad><pad><pad>'

In [26]:
# the attention scores on each token in the first sample in our batch
attn_weights[0, :]

tensor([0.0098, 0.0727, 0.0661, 0.0103, 0.0397, 0.0459, 0.0463, 0.0479, 0.0488,
        0.0457, 0.0540, 0.0507, 0.0334, 0.0370, 0.0561, 0.0430, 0.0171, 0.0456,
        0.0434, 0.0104, 0.0452, 0.0527, 0.0373, 0.0304, 0.0104, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       grad_fn=<SliceBackward0>)

### Results

TLDR: Our model can perform only as best as the baseline does, which is counterintuitive since previous research have showned that the BERT internal structure offers more information than simply predict with the first token from the final layer for some tasks. After some inspection it seems reasonable to believe that our model is trained into some unexpected status: the layer fusion fails to learned a smoothing function that pools across all layers but instead fix to one single layer.

models names e.g.: 
- baseline: the baseline model with ordinary regressor
- mlp_ScalarMix_SelfAttn_pos: MLP + ScalarMixing + Self-Attention + Sinusodial Positional Encoder
- mlp_ScalarMix_AddAttn: MLP + ScalarMixing + Additive Attention
- mlp_ScalarMix_SelfAttn_Lpos: MLP + ScalarMixing + Self-Attention + Learned Positional Encoder

OPEN TRACK
                         model L1  rmse  pearson
 mlp_ScalarMix_SelfAttn_pos_xx es 1.178    0.797
      mlp_ScalarMix_AddAttn_xx es 1.192    0.790
     mlp_ScalarMix_SelfAttn_xx es 1.201    0.784
                mlp_AddAttn_xx es 1.194    0.787
          mlp_SelfAttn_Lpos_xx es 1.234    0.778
              baseline_open_xx es 1.206    0.787
           mlp_SelfAttn_pos_xx es 1.178    0.797
               mlp_SelfAttn_xx es 1.238    0.772
mlp_ScalarMix_SelfAttn_Lpos_xx es 1.185    0.789
          mlp_SelfAttn_Lpos_xx de 1.187    0.787
               mlp_SelfAttn_xx de 1.169    0.785
     mlp_ScalarMix_SelfAttn_xx de 1.179    0.787
 mlp_ScalarMix_SelfAttn_pos_xx de 1.164    0.792
                vib_AddAttn_xx de 1.128    0.801
           mlp_SelfAttn_pos_xx de 1.164    0.792
mlp_ScalarMix_SelfAttn_Lpos_xx de 1.146    0.793
      mlp_ScalarMix_AddAttn_xx de 1.159    0.793
                mlp_AddAttn_xx de 1.161    0.787
              baseline_open_xx de 1.149    0.800
               mlp_SelfAttn_xx cn 1.060    0.787
           mlp_SelfAttn_pos_xx cn 1.038    0.798
mlp_ScalarMix_SelfAttn_Lpos_xx cn 1.021    0.801
      mlp_ScalarMix_AddAttn_xx cn 1.032    0.801
          mlp_SelfAttn_Lpos_xx cn 1.047    0.797
     mlp_ScalarMix_SelfAttn_xx cn 1.061    0.787
              baseline_open_xx cn 1.021    0.804
 mlp_ScalarMix_SelfAttn_pos_xx cn 1.038    0.798
                mlp_AddAttn_xx cn 1.015    0.802